In [ ]:
# DSA 2050 - Step 1: download the 4 datasets for the water-reliability project
# Run in Google Colab (or Jupyter). Takes about 2-10 minutes.
import requests, json, os, zipfile

os.makedirs("water_data", exist_ok=True)
NYC = "https://data.cityofnewyork.us/resource/"

def save(url, params, name):
    r = requests.get(url, params=params, timeout=1200)
    r.raise_for_status()
    with open(f"water_data/{name}", "wb") as f:
        f.write(r.content)
    print(f"OK  {name:36s} {len(r.content)/1e6:7.1f} MB")

# A - NYCHA monthly water bills (meter data)
save(NYC + "66be-66yr.csv", {"$limit": 5_000_000}, "water_bills.csv")

# B - NYCHA Development Data Book (zone data)
save(NYC + "evjd-dqpz.csv", {"$limit": 50_000}, "development_data_book.csv")

# C - 311 "Water System" complaints (outage data), COUNTS only, no addresses.
#     NYC split 311 into two datasets in Dec 2025, so we ask both and join them.
parts = []
for ds, start, end in [("76ig-c548", "2013-01-01", "2019-12-31"),   # 2010-2019 file
                       ("erm2-nwe9", "2020-01-01", "2025-05-31")]:  # 2020-present file
    q = {"$select": "borough, date_trunc_ym(created_date) AS ym, descriptor, count(*) AS n",
         "$where": f"complaint_type = 'Water System' AND created_date between "
                   f"'{start}T00:00:00' and '{end}T23:59:59'",
         "$group": "borough, date_trunc_ym(created_date), descriptor",
         "$limit": 500_000}
    r = requests.get(NYC + ds + ".json", params=q, timeout=1200)
    r.raise_for_status()
    rows = [{"borough": x.get("borough"), "month": x["ym"],
             "descriptor": x.get("descriptor"), "n": x["n"]} for x in r.json()]
    print(f"OK  311 {ds}: {len(rows)} borough-month-type rows")
    parts += rows
with open("water_data/complaints_311_water_system.json", "w") as f:
    json.dump(parts, f)

# D - Daily weather for New York City (Open-Meteo, CC BY 4.0)
save("https://archive-api.open-meteo.com/v1/archive",
     {"latitude": 40.7829, "longitude": -73.9654,
      "start_date": "2013-01-01", "end_date": "2025-05-31",
      "daily": "temperature_2m_mean,temperature_2m_max,temperature_2m_min,"
               "precipitation_sum,snowfall_sum",
      "timezone": "America/New_York"},
     "weather_nyc_daily.json")

# Pack everything into one zip
with zipfile.ZipFile("water_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir("water_data"):
        z.write(f"water_data/{f}", f)
print(f"\nDONE: water_data.zip = {os.path.getsize('water_data.zip')/1e6:.1f} MB")



OK  water_bills.csv                         20.7 MB
OK  development_data_book.csv                0.1 MB
OK  311 76ig-c548: 6551 borough-month-type rows
OK  311 erm2-nwe9: 4701 borough-month-type rows
OK  weather_nyc_daily.json                   0.2 MB

DONE: water_data.zip = 2.2 MB


ModuleNotFoundError: No module named 'google.colab'

In [2]:
import os; print(os.path.abspath("water_data.zip"))

/Users/user/Library/CloudStorage/OneDrive-UnitedStatesInternationalUniversity(USIU)/DSA2050-END-PROJECT/water_data.zip
